# Edmonds-Karp: Max-Flow Pipeline Simulator

Edmonds-Karp finds the maximum amount of flow that can move from a source to a sink through a network with capacities.

Max-flow algorithms began as models for transportation, communication, and logistics networks: roads, pipes, phone lines, supply chains, and matching systems. Edmonds-Karp is a clean teaching version because it uses breadth-first search to choose each augmenting path.

In this notebook, you will build it with small objects: stations, pipes, residual edges, augmenting paths, and a runner that pushes flow until no more can move.

<details>
<summary>Big idea</summary>

Keep finding the shortest augmenting path in the residual graph using BFS. Push as much flow as that path allows. Stop when the sink is no longer reachable.

</details>

## 1. The Mental Model

A flow network is a directed graph with limits:

- **Source**: where flow starts
- **Sink**: where flow must end
- **Capacity**: the maximum flow an edge can carry
- **Flow**: how much is currently being used
- **Residual capacity**: how much more can still move through an edge
- **Augmenting path**: a source-to-sink route with leftover capacity

Edmonds-Karp repeats one move: use BFS to find an augmenting path, then push the path bottleneck.

<details>
<summary>Hint: what is a residual graph?</summary>

The residual graph shows what moves are still possible. It includes forward edges for unused capacity and reverse edges for undoing earlier flow choices.

</details>

## 2. Build the Objects

Implementation plan:

1. `Station` stores a node name.
2. `FlowEdge` stores capacity, current flow, and its reverse edge.
3. `FlowNetwork` stores the residual graph.
4. `FlowStep` records each augmenting path.
5. `EdmondsKarpRunner` owns BFS, flow pushing, and min-cut detection.

<details>
<summary>Implementation hint</summary>

Every real edge gets a reverse edge with capacity `0`. When you push flow forward, the reverse edge gains residual capacity so the algorithm can undo that choice later.

</details>

**Object model.** Define `Station`, the named objects used by the next examples.


In [ ]:
from collections import deque

from dataclasses import dataclass, field

@dataclass(frozen=True, order=True)
class Station:
    name: str

    def __str__(self) -> str:
        return self.name

    def __format__(self, spec: str) -> str:
        return format(self.name, spec)


**Object model.** Define `FlowEdge`, the named objects used by the next examples.


In [ ]:
@dataclass
class FlowEdge:
    start: Station
    end: Station
    capacity: int
    flow: int = 0
    reverse: "FlowEdge | None" = field(default=None, repr=False, compare=False)

    @property
    def residual_capacity(self) -> int:
        return self.capacity - self.flow

    def push(self, amount: int) -> None:
        if amount > self.residual_capacity:
            raise ValueError("Cannot push more than the residual capacity.")
        if self.reverse is None:
            raise ValueError("Residual edge is missing its reverse edge.")

        self.flow += amount
        self.reverse.flow -= amount


**Object model.** Define `FlowNetwork`, the named objects used by the next examples.


In [ ]:
@dataclass
class FlowNetwork:
    adjacency: dict[Station, list[FlowEdge]] = field(default_factory=dict)

    def add_station(self, station: Station) -> None:
        self.adjacency.setdefault(station, [])

    def connect(self, start: Station, end: Station, capacity: int) -> None:
        if capacity < 0:
            raise ValueError("Capacities must be non-negative.")

        self.add_station(start)
        self.add_station(end)

        forward = FlowEdge(start, end, capacity)
        reverse = FlowEdge(end, start, 0)
        forward.reverse = reverse
        reverse.reverse = forward

        self.adjacency[start].append(forward)
        self.adjacency[end].append(reverse)

    def stations(self) -> list[Station]:
        return sorted(self.adjacency)

    def outgoing(self, station: Station) -> list[FlowEdge]:
        return self.adjacency.get(station, [])

    def original_edges(self) -> list[FlowEdge]:
        edges = []
        for station in self.stations():
            edges.extend(edge for edge in self.outgoing(station) if edge.capacity > 0)
        return edges

    def total_flow_from(self, source: Station) -> int:
        return sum(edge.flow for edge in self.outgoing(source) if edge.capacity > 0)

    def describe(self) -> str:
        rows = []
        for edge in self.original_edges():
            rows.append(f"{edge.start:>8} -> {edge.end:<8} capacity {edge.capacity}")
        return "\n".join(rows)

    def flow_summary(self) -> str:
        rows = []
        for edge in self.original_edges():
            rows.append(f"{edge.start:>8} -> {edge.end:<8} flow {edge.flow}/{edge.capacity}")
        return "\n".join(rows)

    def residual_summary(self) -> list[tuple[Station, Station, int]]:
        residuals = []
        for station in self.stations():
            for edge in self.outgoing(station):
                if edge.residual_capacity > 0:
                    residuals.append((edge.start, edge.end, edge.residual_capacity))
        return sorted(residuals, key=lambda item: (item[0].name, item[1].name, item[2]))


**Trace model.** Define `FlowStep`, `FlowResult`, the structure used to capture replayable algorithm state.


In [ ]:
@dataclass
class FlowStep:
    round_number: int
    path: list[tuple[Station, Station]]
    bottleneck: int
    total_flow: int
    flows: list[tuple[Station, Station, int, int]]
    residuals: list[tuple[Station, Station, int]]
    note: str

@dataclass
class FlowResult:
    max_flow: int
    steps: list[FlowStep]
    source_side: list[Station]
    sink_side: list[Station]
    cut_edges: list[tuple[Station, Station, int]]

    @property
    def cut_capacity(self) -> int:
        return sum(capacity for _, _, capacity in self.cut_edges)


**Algorithm engine.** Define `EdmondsKarpRunner`, the class that runs the main simulation or algorithm.


In [ ]:
class EdmondsKarpRunner:
    def __init__(self, network: FlowNetwork, source: Station, sink: Station):
        self.network = network
        self.source = source
        self.sink = sink

    def run(self) -> FlowResult:
        steps: list[FlowStep] = []
        round_number = 0

        while True:
            parents = self._find_augmenting_path()
            if self.sink not in parents:
                break

            path_edges = self._rebuild_path(parents)
            bottleneck = min(edge.residual_capacity for edge in path_edges)

            for edge in path_edges:
                edge.push(bottleneck)

            round_number += 1
            total_flow = self.network.total_flow_from(self.source)
            steps.append(self._snapshot(round_number, path_edges, bottleneck, total_flow))

        source_side = self._reachable_in_residual()
        sink_side = [station for station in self.network.stations() if station not in source_side]
        cut_edges = [
            (edge.start, edge.end, edge.capacity)
            for edge in self.network.original_edges()
            if edge.start in source_side and edge.end in sink_side
        ]

        return FlowResult(
            max_flow=self.network.total_flow_from(self.source),
            steps=steps,
            source_side=source_side,
            sink_side=sink_side,
            cut_edges=cut_edges,
        )

    def _find_augmenting_path(self) -> dict[Station, FlowEdge]:
        parents: dict[Station, FlowEdge] = {}
        seen = {self.source}
        queue = deque([self.source])

        while queue:
            current = queue.popleft()
            for edge in self.network.outgoing(current):
                if edge.residual_capacity <= 0 or edge.end in seen:
                    continue

                parents[edge.end] = edge
                if edge.end == self.sink:
                    return parents

                seen.add(edge.end)
                queue.append(edge.end)

        return parents

    def _rebuild_path(self, parents: dict[Station, FlowEdge]) -> list[FlowEdge]:
        path = []
        current = self.sink

        while current != self.source:
            edge = parents[current]
            path.append(edge)
            current = edge.start

        return path[::-1]

    def _reachable_in_residual(self) -> list[Station]:
        seen = {self.source}
        queue = deque([self.source])

        while queue:
            current = queue.popleft()
            for edge in self.network.outgoing(current):
                if edge.residual_capacity > 0 and edge.end not in seen:
                    seen.add(edge.end)
                    queue.append(edge.end)

        return sorted(seen)

    def _snapshot(self, round_number: int, path_edges: list[FlowEdge], bottleneck: int, total_flow: int) -> FlowStep:
        return FlowStep(
            round_number=round_number,
            path=[(edge.start, edge.end) for edge in path_edges],
            bottleneck=bottleneck,
            total_flow=total_flow,
            flows=[(edge.start, edge.end, edge.flow, edge.capacity) for edge in self.network.original_edges()],
            residuals=self.network.residual_summary(),
            note=f"Push {bottleneck} unit(s) through the BFS path.",
        )


## 3. Create a Tiny Flow Network

Now make a pipeline network. `Source` sends flow through middle stations into `Sink`.

Each edge has a capacity, which is the most flow that pipe can carry.

<details>
<summary>Hint: what should limit the answer?</summary>

The answer is not just the total capacity leaving the source. Middle bottlenecks can limit how much actually reaches the sink.

</details>

In [7]:
source = Station("Source")
a = Station("A")
b = Station("B")
c = Station("C")
sink = Station("Sink")

network = FlowNetwork()
network.connect(source, a, 10)
network.connect(source, b, 8)
network.connect(a, b, 2)
network.connect(a, c, 5)
network.connect(b, c, 7)
network.connect(b, sink, 4)
network.connect(c, sink, 10)

print(network.describe())

       A -> B        capacity 2
       A -> C        capacity 5
       B -> C        capacity 7
       B -> Sink     capacity 4
       C -> Sink     capacity 10
  Source -> A        capacity 10
  Source -> B        capacity 8


## 4. Run Edmonds-Karp

The runner returns a `FlowResult` with:

- `max_flow`: the best total flow from source to sink
- `steps`: one snapshot per augmenting path
- `source_side` and `sink_side`: the final min-cut partition
- `cut_edges`: original edges crossing that cut

<details>
<summary>Quick check</summary>

The max-flow value should equal the min-cut capacity. That is the Max-Flow/Min-Cut Theorem in action.

</details>

In [8]:
runner = EdmondsKarpRunner(network, source, sink)
result = runner.run()

print("Final flows:")
print(network.flow_summary())

print("\nMax flow:", result.max_flow)
print("Source side of min cut:", ", ".join(str(station) for station in result.source_side))
print("Sink side of min cut:", ", ".join(str(station) for station in result.sink_side))
print("Cut edges:", ", ".join(f"{start}->{end}({capacity})" for start, end, capacity in result.cut_edges))
print("Cut capacity:", result.cut_capacity)
print("Theorem check:", result.max_flow == result.cut_capacity)

Final flows:
       A -> B        flow 1/2
       A -> C        flow 5/5
       B -> C        flow 5/7
       B -> Sink     flow 4/4
       C -> Sink     flow 10/10
  Source -> A        flow 6/10
  Source -> B        flow 8/8

Max flow: 14
Source side of min cut: A, B, C, Source
Sink side of min cut: Sink
Cut edges: B->Sink(4), C->Sink(10)
Cut capacity: 14
Theorem check: True


## 5. Replay the Residual Graph

The replay shows each augmenting path, its bottleneck, the total flow so far, and a few residual edges.

<details>
<summary>Hint: how reverse edges show up</summary>

If a path sends flow through `A -> C`, then the residual graph may include `C -> A`. That reverse edge means the algorithm can cancel some earlier flow if a later path needs it.

</details>

**Trace model.** Define `FlowReplay`, the structure used to capture replayable algorithm state.


In [ ]:
class FlowReplay:
    def __init__(self, steps: list[FlowStep]):
        self.steps = steps

    def show(self, limit: int | None = None) -> None:
        selected_steps = self.steps if limit is None else self.steps[:limit]

        for step in selected_steps:
            print(f"Round {step.round_number}: {step.note}")
            print("  path      :", self._format_path(step.path))
            print("  bottleneck:", step.bottleneck)
            print("  total flow:", step.total_flow)
            print("  flows     :", self._format_flows(step.flows))
            print("  residuals :", self._format_residuals(step.residuals[:8]))
            print()

    def _format_path(self, path: list[tuple[Station, Station]]) -> str:
        return " -> ".join(str(start) for start, _ in path) + " -> " + str(path[-1][1])

    def _format_flows(self, flows: list[tuple[Station, Station, int, int]]) -> str:
        return "; ".join(f"{start}->{end} {flow}/{capacity}" for start, end, flow, capacity in flows)

    def _format_residuals(self, residuals: list[tuple[Station, Station, int]]) -> str:
        return "; ".join(f"{start}->{end} {capacity}" for start, end, capacity in residuals) or "none"


**Example state.** Create `replay`, the concrete values used in the next run.


In [ ]:
replay = FlowReplay(result.steps)

replay.show()


## 6. Your Experiments

Try changing one thing at a time:

- Increase `C -> Sink`
- Decrease `Source -> B`
- Add a new middle station
- Add another route into `Sink`

<details>
<summary>Challenge</summary>

Predict whether the max flow changes before you run the cell. Then check whether the min-cut capacity changes too.

</details>

In [10]:
experiment = FlowNetwork()
experiment.connect(source, a, 10)
experiment.connect(source, b, 8)
experiment.connect(a, b, 2)
experiment.connect(a, c, 5)
experiment.connect(b, c, 7)
experiment.connect(b, sink, 4)
experiment.connect(c, sink, 14)

experiment_result = EdmondsKarpRunner(experiment, source, sink).run()

print("Experiment max flow:", experiment_result.max_flow)
print("Experiment cut capacity:", experiment_result.cut_capacity)
print("Theorem check:", experiment_result.max_flow == experiment_result.cut_capacity)
print("\nFinal flows:")
print(experiment.flow_summary())

Experiment max flow: 15
Experiment cut capacity: 15
Theorem check: True

Final flows:
       A -> B        flow 2/2
       A -> C        flow 5/5
       B -> C        flow 6/7
       B -> Sink     flow 4/4
       C -> Sink     flow 11/14
  Source -> A        flow 7/10
  Source -> B        flow 8/8


## Visual Trace + Rigor Studio

**Problem frame.** Move as much flow as possible from source to sink under capacity constraints.

**Interactive animation target.** Animate BFS augmenting paths, bottleneck capacities, residual edges, and the final cut.

**Correctness handle.** Flow conservation holds at every non-source and non-sink node after each augmentation.

**Complexity handle.** O(VE^2), with each BFS choosing a shortest augmenting path in edge count.

**Failure mode to test.** Ignoring residual back edges hides the ability to undo earlier local decisions.

**Studio task.** Lower one capacity and identify the first bottleneck that changes the final max flow.


In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "courseware").exists():
        sys.path.insert(0, str(candidate))
        break

from courseware import AlgorithmPlayer, AlgorithmTrace, TraceStep, render_trace_table

# Convert the implementation above into snapshots:
# trace = AlgorithmTrace("Topic trace")
# trace.append("start", {"your_state": ...}, "What changed?", invariant="What remains true?")
# AlgorithmPlayer(trace, your_renderer).display()
print("Use AlgorithmTrace to turn this notebook's algorithm into a step-by-step visual player.")
